<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-13-capstone-defend-documind/lesson-13.2-capstone-phase-1/notebooks/GCP_Capstone_13.2_CapstonePhase1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13.2 — Build Phase 1: Your Corpus, Your Citations, Your Eval Suite

DocuMind works on a corpus built to make its own tests fail honestly: synthetic tenant documents whose figures deliberately differ, and thirteen real documents held unequally across the tenants. This phase replaces it with yours, and the trap is not technical: build both tenants from the same template and your isolation tests pass whether or not the filter works.


## Cell 1: Done Means These Are True

Ten checks, and the last two are the ones people skip.


In [ ]:
# Phase 1 is done when these are all true. Not before.
#
# Run this against YOUR fork. Every check is a thing you can verify without
# asking anybody.
CHECKS = [
    ('make up completed on your own project',       'the six lean services have URLs: ingest, api, ui, mcp, chat, agent'),
    ('your corpus is ingested',                     '30+ pages, chunks in Firestore'),
    ('a query returns an answer WITH a citation',   'the citation resolves to your document'),
    ('the citation is right',                       'you read the source and it says that'),
    ('a second tenant exists',                      'even if it is you with another email'),
    ('tenant B cannot see tenant A documents',      'you tried it'),
    ('a non-member gets 403',                       'not 401, not an answer'),
    ('golden.jsonl has 30+ rows',                   '5+ of them isolation'),
    ('make eval exits 0',                           'echo $? - do not trust the output text'),
    ('you have made it go RED once',                'on purpose, and you know what did it'),
]
for what, how in CHECKS:
    print(f'  [ ] {what:44} {how}')
print()
print(f'  {len(CHECKS)} checks. The last two are the ones people skip and the two')
print('  the defence asks about, because an eval suite that has never been red is')
print('  indistinguishable from an eval suite that cannot go red.')
print()
print('  And check 4 is not paranoia. A citation that resolves to a real document')
print('  and does not support the answer is the single most damaging failure this')
print('  product can have, because it looks exactly like the thing working.')

## Cell 2: Where make up Stops on a Project You Own

Five differences between the kit&rsquo;s project and yours.


In [ ]:
# make up, on a project you own. Where it stops, and why.
#
# The kit runs. Your project is not the kit's project, and these are the five
# places that difference shows up.
GOTCHAS = [
    ('the Makefile fills the Terraform variables, and four of its defaults are placeholders',
     'PROJECT has no default; ADMIN_EMAILS, BILLING_ACCOUNT_ID, PAGERDUTY_KEY and TFSTATE_BUCKET default to values that plan but cannot apply',
     'pass them on every make up: PROJECT= ADMIN_EMAILS= BILLING_ACCOUNT_ID= TFSTATE_BUCKET= (PAGERDUTY_KEY= if you have one)'),
    ('the state bucket cannot be terraformed',
     'chicken and egg - it holds the state that would create it',
     'create it by hand once: gcloud storage buckets create'),
    ('PROFILE=lean is the default',
     'no Vector Search, BigQuery, Cloud SQL or admin service: Firestore holds the vectors and the rows stay in Logging (12.1, 12.3)',
     'stay on lean; PROFILE=full is one variable away for the day retrieval past ~40k chunks is the thing you are testing, and it bills while it exists'),
    ('IAP needs an OAuth consent screen',
     'a one-time, per-project, console-only step',
     'do it before the demo, not during'),
    ('the audit bucket has a locked retention policy',
     'is_locked = true means it cannot be deleted, ever',
     'project deletion is the only exit - plan for a throwaway project'),
]
for what, why, how in GOTCHAS:
    print(f'  {what}')
    print(f'      {why}')
    print(f'      -> {how}')
    print()
print('  The last one is the one to read twice before you run make up on a')
print('  project you care about. A retention lock is not a setting you can turn')
print('  off; it is a commitment the platform enforces against you as well.')
print()
print('  Use a throwaway project. `gcloud projects delete` is the only teardown')
print('  that reaches zero, and 12.8 says why.')

## Cell 3: A Corpus That Can Fail

Two tenants built from one template cannot demonstrate isolation.


In [ ]:
# Your corpus, not ACME's. This is the first thing the assessor notices.
#
# The demo corpus is synthetic on purpose (no real PII on a shared screen). For
# the capstone you need your own - and "your own" has requirements.
RULES = [
    ('30+ pages',            'below that, retrieval never has to choose'),
    ('at least 2 tenants',   'isolation is untestable with one'),
    ('DIFFERENT answers per tenant',
     'if both handbooks say 60 days, a leak returns the correct answer'),
    ('something unanswerable', 'refusal rows need questions the corpus cannot answer'),
    ('nothing confidential', 'it goes in a fork you will show two strangers'),
]
for rule, why in RULES:
    print(f'  {rule:26} {why}')
print()
print('  Row 3 is the one that quietly ruins capstones. DocuMind\'s own corpus')
print('  gives ACME a Rs 40,000 travel cap and Zeta Rs 25,000 SPECIFICALLY so a')
print('  cross-tenant leak produces a WRONG answer. Build both tenants from the')
print('  same template and your isolation tests pass whether or not the filter')
print('  works - which is the vacuous-assertion trap the rubric scores under')
print('  Evals, waiting for you in your own repo.')
print()
print('  Good sources: your team\'s runbooks, public filings for two companies,')
print('  two open-source projects\' docs. Bad: anything under NDA, anything with')
print('  a real customer in it, anything you would not paste into a slide.')

## Cell 4: Thirty Rows, and What Each Shape Catches

Copying the proportions is fine. Not knowing what each shape is for is not.


In [ ]:
# Thirty rows, four shapes. What each shape is FOR.
#
# Copying DocuMind's proportions is a reasonable start, but only if you know
# what each shape catches - otherwise you write thirty lookups and call it a
# suite.
SHAPES = [
    ('lookup',    13, 'one chunk holds the answer',
     'catches: retrieval got the wrong chunk'),
    ('join',       7, 'two clauses, so packing order and budget matter',
     'catches: the context budget dropped the second one'),
    ('refusal',    5, 'the corpus does not contain it; saying so is correct',
     'catches: the model fabricates rather than declining'),
    ('isolation',  5, 'ask as tenant B, assert tenant A\'s figure never appears',
     'catches: nothing, usually - see below'),
]
print(f'  {"shape":10} {"n":>3}  {"what it is":46} {"what it catches"}')
for shape, n, what, catches in SHAPES:
    print(f'  {shape:10} {n:>3}  {what:46} {catches}')
print()
print('  Row 2 is the underrated one. A join row fails when the context budget')
print('  silently drops the second clause - which happens as your corpus grows,')
print('  looks like a quality regression, and is actually an arithmetic one.')
print()
print('  Row 4 says "catches: nothing, usually" and that is the honest answer.')
print('  If your retriever filters by tenant upstream - and it does - then')
print('  must_not_contain can only fail when the model invents another tenant\'s')
print('  exact figure. Keep the rows; they are cheap and they would catch a')
print('  catastrophe. Just do not call them your isolation test.')
print()
print('  The isolation test that CAN fail asks as a non-member and requires 403.')
print('  That is identity -> tenant, a Firestore lookup, and lookups get loosened.')

---

The rubric and the eight components are published in `course-bibles/capstone-rubric.md`. Score yourself against them before somebody else does.
